I rely on [ShellSage](https://github.com/AnswerDotAI/shell_sage), a shell AI-coding tool developed by [Answer.ai](https://www.answer.ai/), more and more. ShellSage excels at assisting me when writing [Fish shell functions](https://fishshell.com/docs/current/cmds/function.html). I used ShellSage recently to write a Fish shell function to more easily use ShellSage and learned about some useful shell utilities in the process.

## The function

When I invoke ShellSage I put the prompt in single quotes so the Fish shell does not perform any expansion of characters like question marks.

```fish
ssage --mode sassy 'My prompt goes here'
```

I wanted a function that allowed me to,

1. just type `ss` or alternatively `ss` followed by a space and the name of the ShellSage mode I want to use (see "sassy" above), and,
2. upon hitting `Return` get the command prompt filled in with the ShellSage command, the single quotes for my prompt, and my cursor placed neatly in between the quotes so I can just start typing my prompt.

I asked ShellSage to help me write this function (using [Claude 3.7 Sonnet](https://www.anthropic.com/claude/sonnet) as the model), and it showed me something like the following:

```fish
function ss

    # Place the command with the appropriate mode and empty quotes
    commandline -r "ssage ''"

    # Position cursor inside the quotes
    set -l new_pos (math (commandline -C) - 1)
    commandline -C $new_pos

end
```

I had never seen the `commandline` utility before. It does exactly what I need! 

- `commandline -r "ssage ''"` **r**eplaces the current prompt with `ssage ''`.
- The `set` command finds the length of the `ssage` prompt (*eight* characters in this case) and sets `new_pos` equal to one minus that amount (7).
- The second `commandline` command repositions the cursor at `new_pos` which will be one character to the left of the trailing single quote.

Once I understood the function, I added some functionality which lets me adjust the ShellSage mode with an optional argument to `ss`:

```fish
function ss
    # Available modes from ssage
    set -l available_modes default command agent sassy

    # Default to "sassy" mode unless specified otherwise
    set -l mode sassy

    # Check if an argument was provided to override the default mode
    if test (count $argv) -gt 0
        # Validate that the provided mode is allowed
        if contains $argv[1] $available_modes
            set mode $argv[1]
        else
            echo "Invalid mode: $argv[1]"
            echo "Available modes: $available_modes"
            return 1
            fi
        end

    end

    # Place the command with the appropriate mode and empty quotes
    commandline -r "ssage --mode $mode ''"

    # Position cursor inside the quotes, where your creativity can flourish... or whatever
    set -l new_pos (math (commandline -C) - 1)
    commandline -C $new_pos

end
```

## Autocomplete

Now that my function was working as I wanted it, ShellSage helped me add autocomplete for the `mode` argument. Since I have a function called `ss`, Fish will look for autocomplete rules in `~/.config/fish/completions/ss.fish`. To the `ss.fish` file I added the following line (as recommended by ShellSage/Claude):

```fish
complete \
  -c ss \                             # <1>
  -f \                                # <2>
  -a "default command agent sassy" \  # <3>
  -d "mode"                           # <4>
```
1. Defines the name of the command.
2. Disables filenames from completion.
3. This defines the arguments that can be tab-completed.
4. This text will be shown next to each completion option.

And now I can type `ss<space><tab>` and see all the available modes each of which can be autocompleted.